In [1]:
from gliner2 import GLiNER2


In [2]:
pip install transformers datasets torch

  Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
  Using cached multiprocess-0.70.19-py310-none-any.whl (134 kB)
  Using cached dill-0.4.1-py3-none-any.whl (120 kB)
  Using cached pyarrow-23.0.1-cp310-cp310-win_amd64.whl (27.5 MB)
  Using cached xxhash-3.6.0-cp310-cp310-win_amd64.whl (31 kB)
  Using cached aiohttp-3.13.5-cp310-cp310-win_amd64.whl (462 kB)
     -------------------------------------- 41.6/41.6 kB 668.0 kB/s eta 0:00:00
     ---------------------------------------- 87.3/87.3 kB 2.5 MB/s eta 0:00:00
     -------------------------------------- 46.0/46.0 kB 761.3 kB/s eta 0:00:00
     ---------------------------------------- 43.8/43.8 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.3.0
    Uninstalling fsspec-2026.3.0:
      Successfully uninstalled fsspec-2026.3.0
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Load data

In [53]:
import pandas as pd
df = df = pd.read_csv("records.csv")
print(df.head())

                narration  mode    type    category  subcategory
0    FD interest credited  NEFT  Credit  investment  fd_interest
1               FD INT CR  NEFT  Credit  investment  fd_interest
2  Fixed deposit interest  NEFT  Credit  investment  fd_interest
3      FD interest payout  NEFT  Credit  investment  fd_interest
4          Interest on FD  NEFT  Credit  investment  fd_interest


# Create input_text

In [54]:
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

## Lowercase

In [55]:
df["input_text"] = df["input_text"].str.lower()

# Create label (Ground Truth)

In [56]:
df["label"] = df["category"] + "/" + df["subcategory"]

In [57]:
print(df[["input_text", "label"]].head())

                                          input_text                   label
0  narration: fd interest credited | mode: neft |...  investment/fd_interest
1   narration: fd int cr | mode: neft | type: credit  investment/fd_interest
2  narration: fixed deposit interest | mode: neft...  investment/fd_interest
3  narration: fd interest payout | mode: neft | t...  investment/fd_interest
4  narration: interest on fd | mode: neft | type:...  investment/fd_interest


# Check label distribution

In [58]:
print(df["label"].value_counts())

label
investment/fd_interest    8
expense/food              7
income/salary             7
expense/shopping          6
investment/mutual_fund    5
investment/fd_booking     5
expense/bills             5
income/bonus              3
Name: count, dtype: int64


# Define X and y

In [59]:
X = df["input_text"]   
y = df["label"]       

# Train/Test Split

In [60]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # reproducibility
)

# Model Pipeline

In [61]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

## Training

In [62]:
model.fit(X_train, y_train)

,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


## Prediction

In [63]:
y_pred = model.predict(X_test)

In [64]:
from sklearn.metrics import f1_score, fbeta_score, classification_report

# F1 Score (balanced)
f1 = f1_score(y_test, y_pred, average='weighted')

# F2 Score (recall-focused)
f2 = fbeta_score(y_test, y_pred, beta=2, average='weighted')

print("F1 Score:", round(f1, 3))
print("F2 Score:", round(f2, 3))

print("\nDetailed Report:\n")
print(classification_report(y_test, y_pred))

F1 Score: 0.6
F2 Score: 0.594

Detailed Report:

                        precision    recall  f1-score   support

         expense/bills       0.00      0.00      0.00         2
          expense/food       0.00      0.00      0.00         0
      expense/shopping       1.00      0.50      0.67         2
          income/bonus       0.00      0.00      0.00         1
         income/salary       0.50      1.00      0.67         1
 investment/fd_booking       1.00      1.00      1.00         2
investment/fd_interest       1.00      1.00      1.00         2

              accuracy                           0.60        10
             macro avg       0.50      0.50      0.48        10
          weighted avg       0.65      0.60      0.60        10



c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

## Accuracy

In [65]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.6


In [66]:
print(df[["narration", "label"]])

                    narration                   label
0        FD interest credited  investment/fd_interest
1                   FD INT CR  investment/fd_interest
2      Fixed deposit interest  investment/fd_interest
3          FD interest payout  investment/fd_interest
4              Interest on FD  investment/fd_interest
5        FD interest received  investment/fd_interest
6        FD maturity interest  investment/fd_interest
7            Bank FD interest  investment/fd_interest
8                  FD booking   investment/fd_booking
9              Open FD online   investment/fd_booking
10  FD created via netbanking   investment/fd_booking
11             New FD booking   investment/fd_booking
12          FD account opened   investment/fd_booking
13                  SIP debit  investment/mutual_fund
14         MF SIP installment  investment/mutual_fund
15                SIP payment  investment/mutual_fund
16      Mutual fund SIP debit  investment/mutual_fund
17      Monthly SIP deductio

In [67]:
print(df["label"].value_counts())

label
investment/fd_interest    8
expense/food              7
income/salary             7
expense/shopping          6
investment/mutual_fund    5
investment/fd_booking     5
expense/bills             5
income/bonus              3
Name: count, dtype: int64


In [68]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[0 2 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 1 1 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 0 0 2 0]
 [0 0 0 0 0 0 2]]


In [69]:
vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()

for i, label in enumerate(classifier.classes_):
    top_features = sorted(
        zip(classifier.coef_[i], feature_names),
        reverse=True
    )[:5]
    
    print(f"\nTop words for {label}:")
    for coef, word in top_features:
        print(word)


Top words for expense/bills:
bill
recharge
mobile
internet
water

Top words for expense/food:
payment
upi
order
swiggy
restaurant

Top words for expense/shopping:
shopping
purchase
upi
ecommerce
clothing

Top words for income/bonus:
bonus
yearly
credited
neft
credit

Top words for income/salary:
salary
credit
neft
company
payroll

Top words for investment/fd_booking:
netbanking
fd
open
new
booking

Top words for investment/fd_interest:
interest
fd
neft
credit
int

Top words for investment/mutual_fund:
sip
auto_debit
debit
deduction
mf


## Debug

In [70]:
for i in range(len(X_test)):
    print("Input:", X_test.iloc[i])
    print("Actual:", y_test.iloc[i])
    print("Predicted:", y_pred[i])
    print("-----")

Input: narration: performance bonus | mode: neft | type: credit
Actual: income/bonus
Predicted: income/salary
-----
Input: narration: amazon purchase | mode: upi | type: debit
Actual: expense/shopping
Predicted: expense/shopping
-----
Input: narration: flipkart order | mode: upi | type: debit
Actual: expense/shopping
Predicted: expense/food
-----
Input: narration: utility payment | mode: upi | type: debit
Actual: expense/bills
Predicted: expense/food
-----
Input: narration: salary neft | mode: neft | type: credit
Actual: income/salary
Predicted: income/salary
-----
Input: narration: electricity bill payment | mode: upi | type: debit
Actual: expense/bills
Predicted: expense/food
-----
Input: narration: interest on fd | mode: neft | type: credit
Actual: investment/fd_interest
Predicted: investment/fd_interest
-----
Input: narration: fd account opened | mode: netbanking | type: debit
Actual: investment/fd_booking
Predicted: investment/fd_booking
-----
Input: narration: fd booking | mode: 

## Confidence

In [71]:
probs = model.predict_proba(X_test)
for i in range(len(X_test)):
    print("Prediction:", model.classes_[probs[i].argmax()])
    print("Confidence:", round(probs[i].max(), 2))

Prediction: income/salary
Confidence: 0.23
Prediction: expense/shopping
Confidence: 0.28
Prediction: expense/food
Confidence: 0.42
Prediction: expense/food
Confidence: 0.48
Prediction: income/salary
Confidence: 0.47
Prediction: expense/food
Confidence: 0.33
Prediction: investment/fd_interest
Confidence: 0.51
Prediction: investment/fd_booking
Confidence: 0.26
Prediction: investment/fd_booking
Confidence: 0.27
Prediction: investment/fd_interest
Confidence: 0.51


In [72]:
def confidence_level(conf):
    if conf > 0.75:
        return "High"
    elif conf > 0.5:
        return "Medium"
    else:
        return "Low"

In [73]:
from sklearn.metrics import classification_report, f1_score

print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

print("\nDetailed Report:\n")
print(classification_report(y_test, y_pred))

F1 Score: 0.6

Detailed Report:

                        precision    recall  f1-score   support

         expense/bills       0.00      0.00      0.00         2
          expense/food       0.00      0.00      0.00         0
      expense/shopping       1.00      0.50      0.67         2
          income/bonus       0.00      0.00      0.00         1
         income/salary       0.50      1.00      0.67         1
 investment/fd_booking       1.00      1.00      1.00         2
investment/fd_interest       1.00      1.00      1.00         2

              accuracy                           0.60        10
             macro avg       0.50      0.50      0.48        10
          weighted avg       0.65      0.60      0.60        10



c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

# Output

In [74]:
results = []
for i in range(len(df)):
    input_text = df["input_text"].iloc[i]
    probs = model.predict_proba([input_text])[0]
    pred_label = model.classes_[probs.argmax()]
    confidence = probs.max()
    category, subcategory = pred_label.split("/", 1)
    results.append({
    "narration": df["narration"].iloc[i],
    "mode": df["mode"].iloc[i],
    "type": df["type"].iloc[i],
    "ground_truth": f"{category}/{subcategory}",
    "confidence": round(confidence, 2),
    "confidence_percent": f"{round(confidence*100, 1)}%",
    "confidence_level": confidence_level(confidence)
})
result_df = pd.DataFrame(results)
print(result_df.to_string())

                    narration        mode    type            ground_truth  confidence confidence_percent confidence_level
0        FD interest credited        NEFT  Credit  investment/fd_interest        0.46              45.7%              Low
1                   FD INT CR        NEFT  Credit  investment/fd_interest        0.37              37.1%              Low
2      Fixed deposit interest        NEFT  Credit  investment/fd_interest        0.40              40.2%              Low
3          FD interest payout        NEFT  Credit  investment/fd_interest        0.51              51.4%           Medium
4              Interest on FD        NEFT  Credit  investment/fd_interest        0.51              51.4%           Medium
5        FD interest received        NEFT  Credit  investment/fd_interest        0.48              47.9%              Low
6        FD maturity interest        NEFT  Credit  investment/fd_interest        0.48              47.9%              Low
7            Bank FD int